Processar o arquivo de base com as perguntas.

In [ ]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-04 09:54:41--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9811 (9.6K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]   9.58K  --.-KB/s    in 0s      

2026-04-04 09:54:41 (21.1 MB/s) - ‘questions.ipynb’ saved [9811/9811]



Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [16]:
# Instalar ou atualizar as bibliotecas necessárias.
!pip install -q -U openai bert-score
# !pip install -q -U google-genai google-auth

# Importar bibliotecas.
import pandas as pd
import os
from openai import OpenAI
from google import genai
from google.colab import userdata
from bert_score import score

Funcões dos clientes das IAs, modelos e realização de consultas.

In [ ]:
# Iniciar o cliente da IA
def client_ai_instance(groq_api_key):
  # Inicializar o cliente da API.
  client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=groq_api_key
  )
  return client_ai

def markdown_prepare(papel, categoria, contexto, dificuldade, instrucao):
  # Preparação do prompt em Markdown
  prompt_usuario = f"""
  ### Papel:
  {papel}

  ### Área de atuação:
  {categoria}

  ### Contexto:
  {contexto}

  ### Nível de Dificuldade:
  Com valores de 1 a 4, onde 1 indica o mais simples e 4 o mais complexo.
  {dificuldade}

  ### Instrução:
  {instrucao}
  """

  return prompt_usuario

def questions_submit(client_ai, model_id, df_questions):
    # Criar lista vazia para guardar as respostas.
    responses = []

    # Repetição para percorrer todo Dataframe.
    for index, row in df_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      questao = row['question']
      papel = row['system']
      categoria = row['category']
      contexto = row['statement']
      dificuldade = row['level']
      instrucao = row['turns']

      # Preencher prompt do usuário.
      prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

      # Realizar consulta a IA.
      completion = client_ai.chat.completions.create(
          model= model_id,
          messages=[
            # Por questões de sintaxes informo a role, pois é um campo obrigatório,
            # porém o conteúdo é somente o do Markdown.
            {"role": "user", "content": prompt_usuario}
          ],
          max_tokens=1024,
          temperature=0.1 # Conservador.
      )
      response = completion.choices[0].message.content

      # Armazenar o resultado em uma lista.
      responses.append({"question": questao, "response": response})
      print(f"Questão {questao} processada com sucesso.")

    # Fechar conexão com a IA (somente se você a usou ativamente)
    client_ai.close()

    # por motivo de performance as repostas foram adicionadas a uma lista.
    # No retorno a lista é convertida para um dataframe.
    return pd.DataFrame(responses)

Realizar consulta usando o modelo llama 3 de 70 bilhões de parâmetros.

In [ ]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# Tal chave foi previamente criada no Console do Groq, site: console.groq.com
groq_api_key = userdata.get('groq_api_key')

# Modelo llama 3 de 70 bilhões de parâmetros.
model_id = 'llama-3.3-70b-versatile'

# Dataframe com as respostas do primeiro modelo
df_llama_response = questions_submit(client_ai_instance(groq_api_key), model_id, df_my_questions)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.
Questão 35 processada com sucesso.
Questão 36 processada com sucesso.
Questão 37 processada com sucesso.
Questão 38 processada com sucesso.
Questão 39 processada com sucesso.
Questão 40 processada com sucesso.


Modelo qwen 3 de 32 bilhões de parâmetros.

In [ ]:
# Modelo llama 4 scout, mais leve, de 17 bilhões de parâmetros.
model_id = 'meta-llama/llama-4-scout-17b-16e-instruct'

# Dataframe com as respostas do primeiro modelo
df_llama4_response = questions_submit(client_ai_instance(groq_api_key), model_id, df_my_questions)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.
Questão 35 processada com sucesso.
Questão 36 processada com sucesso.
Questão 37 processada com sucesso.
Questão 38 processada com sucesso.
Questão 39 processada com sucesso.
Questão 40 processada com sucesso.


Específico para o gemini

In [19]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# O uso do Secrets é uma alternativa para que chave de API não fique exposta no código.
# Tal chave previamente criada no Google AI Studio.
# Observação: o nome da chave definido no Google AI Studio precisa ser exatamente o mesmo cadastrado no Secrets.
#             inclusive com diferenciação de letra maiúscula e minúscula.
google_api_key = userdata.get('google_api_key')

# O modelo escolhi do para rodar é o Gemini da Google em nuvém preview de 2026.
model_id = 'gemini-3.1-flash-lite-preview'

# Instancia o modelo.
client_ai = genai.Client(api_key=google_api_key)

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gemini_responses = []

# Repetição que percorre todo Dataframe.
for index, row in df_my_questions.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    dificuldade = row['level']
    instrucao = row['turns']

    # Preparação do prompt em Markdown.
    prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

    # Chamada simples para a API da Google em nuvem.
    response = client_ai.models.generate_content(
        model=model_id,
        contents=prompt_usuario,
        config={
            "temperature": 0.1,  # Conservador.
            "max_output_tokens": 1024
        }
    )

    # Armazenar o resultado em uma lista.
    gemini_responses.append({"question": questao, "response": response.candidates[0].content.parts[0].text})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gemini_response = pd.DataFrame(gemini_responses)

,question,response
0,31,sdk_http_response=HttpResponse(\n headers=<di...
1,32,sdk_http_response=HttpResponse(\n headers=<di...
2,33,sdk_http_response=HttpResponse(\n headers=<di...
3,34,sdk_http_response=HttpResponse(\n headers=<di...
4,35,sdk_http_response=HttpResponse(\n headers=<di...
5,36,sdk_http_response=HttpResponse(\n headers=<di...
6,37,sdk_http_response=HttpResponse(\n headers=<di...
7,38,sdk_http_response=HttpResponse(\n headers=<di...
8,39,sdk_http_response=HttpResponse(\n headers=<di...
9,40,sdk_http_response=HttpResponse(\n headers=<di...


In [56]:
# Função para juntar dois Dataframes por um campo chave e opcionalmente selecionar colunas.
def merge_dataframes(df1, df2, key, columns=None):
  merged_df = pd.merge(df1, df2, on=key, how='inner')
  if columns is not None:
    return merged_df[columns]
  return merged_df

# Renomear colunas dos dataframes para melhor legibilidade.
df_llama_response.rename(columns={'response': 'llama3'}, inplace=True)
df_llama4_response.rename(columns={'response': 'llama4'}, inplace=True)
df_gemini_response.rename(columns={'response': 'gemini'}, inplace=True)
key = 'question'

# Realizar o primeiro inner join entre llama_responses e llama4_responses
df_llamas = merge_dataframes(df_llama_response, df_llama4_response, key)

# Realizar o segundo inner join com df_gemini_response
df_llamas_gemini = merge_dataframes(df_llamas, df_gemini_response, key)

# Junção com o Dataframe de respostas das IAs com o das perguntas, selecionando a coluna choices da linha de guia de respostas.
colums = ['question', 'llama3', 'llama4', 'gemini', 'choices']
df_ias = merge_dataframes(df_llamas_gemini, df_my_questions, key, colums)

,question,llama3,llama4,gemini,choices
0,31,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,"A medida judicial cabível é a ação anulatória,..."
1,32,O prazo para oferta dos embargos à execução fi...,O prazo para oferta dos embargos à execução fi...,O prazo para a oferta dos Embargos à Execução ...,O prazo para oferta dos embargos à execução fi...
2,33,O Imposto Sobre Serviços (ISS) é um tributo mu...,## Resposta\n\nO Imposto Sobre Serviços (ISS) ...,O Imposto Sobre Serviços de Qualquer Natureza ...,O ISS de jardinagem é devido ao Município Beta...
3,34,A exigência de depósito do montante integral c...,A exigência de depósito do montante integral c...,"Não, a exigência de depósito do montante integ...","Não, pois é inconstitucional a exigência de de..."
4,35,O argumento apresentado pelo departamento jurí...,## Resposta\n\n### Item 1: Correção do Argumen...,"Não, o argumento da sociedade empresária não e...","Não está correto, porque a entrega de declaraç..."
5,36,### RECURSO DE AGRAVO DE INSTRUMENTO\n\n**EXCE...,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,EXCELENTÍSSIMO SENHOR DOUTOR DESEMBARGADOR PRE...,O(a) examinando(a) deve apresentar recurso de ...
6,37,A responsabilização administrativa de uma soci...,A responsabilização administrativa de pessoas ...,Para a responsabilização administrativa da soc...,Não. A responsabilização administrativa por at...
7,38,A cobrança efetuada pelo órgão responsável par...,A questão apresentada envolve a interpretação ...,"Não, a cobrança efetuada pelo órgão público é ...",Não. A submissão e o processamento de pedido d...
8,39,A possibilidade de utilização do sistema de re...,"Sim, é possível a utilização do sistema de reg...","Sim, é possível a utilização do Sistema de Reg...",Sim. A Administração poderá contratar a execuç...
9,40,A aprovação de Iná no concurso público para a ...,A aprovação de Iná no concurso público para a ...,Não. A aprovação de Iná no concurso público pa...,"Não. Iná foi aprovada para emprego público, ao..."


In [40]:
from google.colab import drive

# Montar o Google Drive
drive.mount('/content/drive')

# Definir o caminho no Google Drive onde o arquivo será salvo
output_path = '/content/drive/My Drive/df_ias.jsonl'

# Salvar o DataFrame df_ias como um arquivo .jsonl
# 'orient='records'' para ter um JSON por linha, e 'lines=True' para o formato .jsonl
df_ias.to_json(output_path, orient='records', lines=True)

print(f"DataFrame df_ias exportado com sucesso para {output_path}")

Mounted at /content/drive
DataFrame df_ias exportado com sucesso para /content/drive/My Drive/df_ias.jsonl


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def calculate_similarity(text1, text2):
    """
    Calculates the cosine similarity between two text strings after TF-IDF vectorization.
    Returns 0 if one of the texts is empty or invalid.
    """
    if not isinstance(text1, str) or not isinstance(text2, str) or not text1.strip() or not text2.strip():
        return 0.0

    vectorizer = TfidfVectorizer().fit([text1, text2])
    tfidf_text1 = vectorizer.transform([text1])
    tfidf_text2 = vectorizer.transform([text2])
    return cosine_similarity(tfidf_text1, tfidf_text2)[0][0]

# Apply the similarity function to create a new column
df_ias['llama3_vs_choices_similarity'] = df_ias.apply(lambda row: calculate_similarity(row['llama3'], row['choices']), axis=1)

# Display the DataFrame with the new similarity column
display(df_ias[['question', 'llama3', 'choices', 'llama3_vs_choices_similarity']].head())

# Display descriptive statistics for the similarity score
print("\nDescriptive statistics for Llama3 vs Choices Similarity:")
display(df_ias['llama3_vs_choices_similarity'].describe())

,question,llama3,choices,llama3_vs_choices_similarity
0,31,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,"A medida judicial cabível é a ação anulatória,...",0.658986
1,32,O prazo para oferta dos embargos à execução fi...,O prazo para oferta dos embargos à execução fi...,0.412022
2,33,O Imposto Sobre Serviços (ISS) é um tributo mu...,O ISS de jardinagem é devido ao Município Beta...,0.347458
3,34,A exigência de depósito do montante integral c...,"Não, pois é inconstitucional a exigência de de...",0.341677
4,35,O argumento apresentado pelo departamento jurí...,"Não está correto, porque a entrega de declaraç...",0.233079



Descriptive statistics for Llama3 vs Choices Similarity:


,llama3_vs_choices_similarity
count,10.000000
mean,0.420135
std,0.151595
min,0.193906
25%,0.343122
50%,0.419530
75%,0.517919
max,0.658986


In [64]:
def calculate_bertscore(df_ias):
  responses = []
  for index, row in df_ias.iterrows():
    candidatos = [[df_ias['llama3'], df_ias['llama4'], df_ias['gemini']]]
    P, R, F1 = score(candidatos, referencias, lang="pt", verbose=True)
    referencias = [df_ias['choices']
    similaridade = F1.mean().item()
    responses.append({"question": questao, "similaridade": similaridade})
    print(f"Nota de Similaridade Semântica (F1): {similaridade}")
  return pd.DataFrame(responses)

calculate_bertscore(df_ias)

SyntaxError: '[' was never closed (2986868960.py, line 6)

**Reasoning**:
I need to define the `calculate_bert_f1` function as specified in the instructions, including input validation, type conversion, and the call to `bert_score.score` to calculate F1 scores. This function will encapsulate the BERTScore logic and handle potential input issues.



In [57]:
def calculate_bert_f1(candidates, references):
    """
    Computes BERTScore F1 scores between lists of candidate and reference texts.

    Args:
        candidates (list): A list of candidate response strings.
        references (list): A list of reference answer strings.

    Returns:
        list: A list of F1 scores from BERTScore, or an empty list if inputs are invalid.
    """
    if not isinstance(candidates, list) or not isinstance(references, list):
        print("Error: Both 'candidates' and 'references' must be lists.")
        return []

    if not candidates or not references:
        print("Warning: One or both input lists are empty. Returning an empty list.")
        return []

    if len(candidates) != len(references):
        print("Error: 'candidates' and 'references' lists must have the same length.")
        return []

    # Ensure all elements are strings
    processed_candidates = []
    for item in candidates:
        if isinstance(item, (str, int, float)):
            processed_candidates.append(str(item))
        else:
            processed_candidates.append("") # Convert to empty string for invalid types

    processed_references = []
    for item in references:
        if isinstance(item, (str, int, float)):
            processed_references.append(str(item))
        else:
            processed_references.append("") # Convert to empty string for invalid types

    # Calculate BERTScore
    try:
        P, R, F1 = score(processed_candidates, processed_references, lang="pt", verbose=False)
        return F1.tolist()
    except Exception as e:
        print(f"Error calculating BERTScore: {e}")
        return []

print("BERTScore F1 calculation function defined.")

BERTScore F1 calculation function defined.


**Reasoning**:
The `calculate_bert_f1` function has been defined. Now, I need to apply this function to the `df_ias` DataFrame to compute BERTScore F1 scores for each AI model's response ('llama3', 'llama4', 'gemini') against the reference answers ('choices'). Then, I will display the head of the DataFrame with the new columns and their descriptive statistics as outlined in the subtask instructions.



In [59]:
import pandas as pd
from bert_score import score

# The dataframe df_ias is already available from previous steps.
# Now, calculate BERTScore for Llama3 responses against 'choices'.
print("Calculating BERTScore F1 for Llama3...")
df_ias['llama3_bert_f1'] = calculate_bert_f1(df_ias['llama3'].tolist(), df_ias['choices'].tolist())
print("Llama3 BERTScore F1 calculation complete.")

# Calculate BERTScore for Llama4 responses against 'choices'.
print("\nCalculating BERTScore F1 for Llama4...")
df_ias['llama4_bert_f1'] = calculate_bert_f1(df_ias['llama4'].tolist(), df_ias['choices'].tolist())
print("Llama4 BERTScore F1 calculation complete.")

# Calculate BERTScore for Gemini responses against 'choices'.
print("\nCalculating BERTScore F1 for Gemini...")
df_ias['gemini_bert_f1'] = calculate_bert_f1(df_ias['gemini'].tolist(), df_ias['choices'].tolist())
print("Gemini BERTScore F1 calculation complete.")

# Display the head of the DataFrame with new BERTScore F1 columns
print("\nDataFrame head with new BERTScore F1 columns:")
display(df_ias[['question', 'llama3_bert_f1', 'llama4_bert_f1', 'gemini_bert_f1']].head())

# Display descriptive statistics for the BERTScore F1 values for each model
print("\nDescriptive statistics for BERTScore F1 (Llama3):")
display(df_ias['llama3_bert_f1'].describe())

print("\nDescriptive statistics for BERTScore F1 (Llama4):")
display(df_ias['llama4_bert_f1'].describe())

print("\nDescriptive statistics for BERTScore F1 (Gemini):")
display(df_ias['gemini_bert_f1'].describe())

Calculating BERTScore F1 for Llama3...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Llama3 BERTScore F1 calculation complete.

Calculating BERTScore F1 for Llama4...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

,llama3_bert_f1
count,10.000000
mean,0.576328
std,0.069495
min,0.520284
25%,0.527205
50%,0.553777
75%,0.583071
max,0.705199



Descriptive statistics for BERTScore F1 (Llama4):


,llama4_bert_f1
count,10.000000
mean,0.590748
std,0.065442
min,0.523658
25%,0.555794
50%,0.568386
75%,0.586078
max,0.711687



Descriptive statistics for BERTScore F1 (Gemini):


,gemini_bert_f1
count,10.000000
mean,0.598864
std,0.058691
min,0.539732
25%,0.563565
50%,0.576743
75%,0.634570
max,0.699980


**Reasoning**:
The subtask requires calculating BERTScore F1 for 'llama3' responses against 'choices' and storing it in a new column. Although the previous cell `fdd3ebfd` attempts this, its execution might have been incomplete or problematic as suggested by the BERTModel load report in the standard error. To ensure this specific part of the subtask is completed, I will generate a new code cell that explicitly calculates and assigns the `llama3_bert_f1` scores.



In [65]:
print("Calculating BERTScore F1 for Llama3...")
df_ias['llama3_bert_f1'] = calculate_bert_f1(df_ias['llama3'].tolist(), df_ias['choices'].tolist())
print("Llama3 BERTScore F1 calculation complete.")

# Display the head of the DataFrame with the new BERTScore F1 column
print("\nDataFrame head with new BERTScore F1 column for Llama3:")
display(df_ias[['question', 'llama3', 'choices', 'llama3_bert_f1']].head())

# Display descriptive statistics for the BERTScore F1 values for Llama3
print("\nDescriptive statistics for BERTScore F1 (Llama3):")
display(df_ias['llama3_bert_f1'].describe())

Calculating BERTScore F1 for Llama3...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Llama3 BERTScore F1 calculation complete.

DataFrame head with new BERTScore F1 column for Llama3:


,question,llama3,choices,llama3_bert_f1
0,31,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,"A medida judicial cabível é a ação anulatória,...",0.698153
1,32,O prazo para oferta dos embargos à execução fi...,O prazo para oferta dos embargos à execução fi...,0.589503
2,33,O Imposto Sobre Serviços (ISS) é um tributo mu...,O ISS de jardinagem é devido ao Município Beta...,0.552380
3,34,A exigência de depósito do montante integral c...,"Não, pois é inconstitucional a exigência de de...",0.563776
4,35,O argumento apresentado pelo departamento jurí...,"Não está correto, porque a entrega de declaraç...",0.527438



Descriptive statistics for BERTScore F1 (Llama3):


,llama3_bert_f1
count,10.000000
mean,0.576328
std,0.069495
min,0.520284
25%,0.527205
50%,0.553777
75%,0.583071
max,0.705199


# Task
To implement the BERTScore function, I'll define a Python function that takes a list of candidate responses and a list of reference answers as input. This function will encapsulate the `bert_score.score` call and return the F1 scores.

```python
import pandas as pd
from bert_score import score

def calculate_bert_f1(candidates, references):
    """
    Calculates BERTScore F1 between a list of candidate responses and a list of reference answers.

    Args:
        candidates (list): A list of candidate response strings.
        references (list): A list of reference answer strings.

    Returns:
        list: A list of F1 scores for each candidate-reference pair.
              Returns an empty list if inputs are invalid or empty.
    """
    if not isinstance(candidates, list) or not isinstance(references, list):
        print("Error: candidates and references must be lists.")
        return []
    if not candidates or not references:
        print("Warning: Empty lists provided for candidates or references.")
        return []
    if len(candidates) != len(references):
        print("Error: The number of candidates must match the number of references.")
        return []

    # Ensure all inputs are strings, replace non-strings with empty string to avoid errors
    processed_candidates = [str(c) if isinstance(c, (str, int, float)) else "" for c in candidates]
    processed_references = [str(r) if isinstance(r, (str, int, float)) else "" for r in references]

    # Filter out empty strings from processed_candidates and processed_references
    # If a candidate or reference is empty, BERTScore might produce NaN or errors.
    # For now, we'll keep them as empty strings and handle potential NaNs from BERTScore.
    # BERTScore itself handles empty strings gracefully by returning 0 for P, R, F1.

    # Calculate BERTScore
    # We use lang="pt" for Portuguese, assuming the context of the notebook.
    # verbose=False to suppress detailed output for each score calculation.
    P, R, F1 = score(processed_candidates, processed_references, lang="pt", verbose=False)

    return F1.tolist()

# The dataframe df_ias is already available from previous steps.
# Now, calculate BERTScore for Llama3 responses against 'choices'.
print("Calculating BERTScore F1 for Llama3...")
df_ias['llama3_bert_f1'] = calculate_bert_f1(df_ias['llama3'].tolist(), df_ias['choices'].tolist())
print("Llama3 BERTScore F1 calculation complete.")

# Calculate BERTScore for Llama4 responses against 'choices'.
print("\nCalculating BERTScore F1 for Llama4...")
df_ias['llama4_bert_f1'] = calculate_bert_f1(df_ias['llama4'].tolist(), df_ias['choices'].tolist())
print("Llama4 BERTScore F1 calculation complete.")

# Calculate BERTScore for Gemini responses against 'choices'.
print("\nCalculating BERTScore F1 for Gemini...")
df_ias['gemini_bert_f1'] = calculate_bert_f1(df_ias['gemini'].tolist(), df_ias['choices'].tolist())
print("Gemini BERTScore F1 calculation complete.")

# Display the head of the DataFrame with new BERTScore F1 columns
print("\nDataFrame head with new BERTScore F1 columns:")
display(df_ias[['question', 'llama3_bert_f1', 'llama4_bert_f1', 'gemini_bert_f1']].head())

# Display descriptive statistics for the BERTScore F1 values for each model
print("\nDescriptive statistics for BERTScore F1 (Llama3):")
display(df_ias['llama3_bert_f1'].describe())

print("\nDescriptive statistics for BERTScore F1 (Llama4):")
display(df_ias['llama4_bert_f1'].describe())

print("\nDescriptive statistics for BERTScore F1 (Gemini):")
display(df_ias['gemini_bert_f1'].describe())
```

## Implement BERTScore Function

### Subtask:
Define a Python function `calculate_bert_f1` to compute BERTScore F1 scores between lists of candidate and reference texts, including error handling for invalid inputs.


# Task
To visualize the distribution of BERTScore F1 values, I'll create a box plot for each model (Llama3, Llama4, and Gemini). This will allow for a clear comparison of their performance.

Then, I'll provide a summary of the BERTScore results, identifying the best-performing model and highlighting any significant observations from the scores and the visualization.

## Calculate BERTScore for Llama3

### Subtask:
Apply the BERTScore function to compare the 'llama3' responses with the 'choices' in `df_ias`. Store the resulting F1 scores in a new column named `llama3_bert_f1` in `df_ias`.


**Reasoning**:
The subtask explicitly requests applying the `calculate_bert_f1` function to compare 'llama3' responses with 'choices' and storing the F1 scores in a new column `llama3_bert_f1` in `df_ias`. I will generate a code block to perform these actions as instructed.



In [66]:
print("Calculating BERTScore F1 for Llama3...")
df_ias['llama3_bert_f1'] = calculate_bert_f1(df_ias['llama3'].tolist(), df_ias['choices'].tolist())
print("Llama3 BERTScore F1 calculation complete.")

Calculating BERTScore F1 for Llama3...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Llama3 BERTScore F1 calculation complete.


**Reasoning**:
The previous steps successfully calculated the BERTScore F1 for Llama3. Now, I will calculate the BERTScore F1 for Llama4 and Gemini responses against the 'choices' as per the overall task, then display the head of the DataFrame with all new BERTScore F1 columns and their descriptive statistics.



In [67]:
print("Calculating BERTScore F1 for Llama4...")
df_ias['llama4_bert_f1'] = calculate_bert_f1(df_ias['llama4'].tolist(), df_ias['choices'].tolist())
print("Llama4 BERTScore F1 calculation complete.")

print("\nCalculating BERTScore F1 for Gemini...")
df_ias['gemini_bert_f1'] = calculate_bert_f1(df_ias['gemini'].tolist(), df_ias['choices'].tolist())
print("Gemini BERTScore F1 calculation complete.")

# Display the head of the DataFrame with new BERTScore F1 columns
print("\nDataFrame head with new BERTScore F1 columns:")
display(df_ias[['question', 'llama3_bert_f1', 'llama4_bert_f1', 'gemini_bert_f1']].head())

# Display descriptive statistics for the BERTScore F1 values for each model
print("\nDescriptive statistics for BERTScore F1 (Llama3):")
display(df_ias['llama3_bert_f1'].describe())

print("\nDescriptive statistics for BERTScore F1 (Llama4):")
display(df_ias['llama4_bert_f1'].describe())

print("\nDescriptive statistics for BERTScore F1 (Gemini):")
display(df_ias['gemini_bert_f1'].describe())

Calculating BERTScore F1 for Llama4...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Llama4 BERTScore F1 calculation complete.

Calculating BERTScore F1 for Gemini...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

,llama3_bert_f1
count,10.000000
mean,0.576328
std,0.069495
min,0.520284
25%,0.527205
50%,0.553777
75%,0.583071
max,0.705199



Descriptive statistics for BERTScore F1 (Llama4):


,llama4_bert_f1
count,10.000000
mean,0.590748
std,0.065442
min,0.523658
25%,0.555794
50%,0.568386
75%,0.586078
max,0.711687



Descriptive statistics for BERTScore F1 (Gemini):


,gemini_bert_f1
count,10.000000
mean,0.598864
std,0.058691
min,0.539732
25%,0.563565
50%,0.576743
75%,0.634570
max,0.699980
